#### INSTALL & IMPORTS

In [ ]:
!pip install ultralytics
!pip install gTTS pydub

import os
import cv2
import json
import gzip
import numpy as np
import torch
import time

from PIL import ImageFont, ImageDraw, Image
from gtts import gTTS
from pydub import AudioSegment
from ultralytics import YOLO

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.0
    Uninstalling click-8.4.0:
      Successfully uninstalled click-8.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.25.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.27.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.c

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


#### PHASE 1: ANALYZE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ================= CONFIG =================
class ThesisConfig:
    DEBUG = False
    DATA_PATH = "/content/drive/MyDrive/KLTN/my_data_ver2"
    #DATA_PATH = "/content/drive/MyDrive/KLTN/dataset/data5/left"

    IMG_DIR   = DATA_PATH + "/video_frames"
    DEPTH_DIR = DATA_PATH + "/depth_maps"
    SEG_DIR   = DATA_PATH + "/segmentation_masks"

    OUT_DIR   = "/content/drive/MyDrive/KLTN/output"
    LABEL_COCO = "/content/drive/MyDrive/KLTN/labels_coco_vi.json"

    # Resize toàn bộ ảnh về cùng file
    WIDTH, HEIGHT = 1242, 736

    # Ngưỡng tin cậy
    YOLO_CONF = 0.4

    USE_DEPTH = True
    USE_SEGMENTATION = True

    # Render YOLO mỗi 3 frame
    YOLO_RENDER_SKIP_INTERVAL = 3

    DATASET_TYPE = "ADE20K"

    SEGMENTATION_SEMANTICS = {

          "walkable": [
              "floor",
              "road",
              "sidewalk",
              "stairs",
              "terrain",
              "crosswalk",
              "walkable"
          ],

          "stairs": [
              "stairs"
          ]
      }

    SEGMENTATION_ID2LABEL = {

        "SANPO": {

            3: "sidewalk",
            5: "crosswalk",
            6: "road",
            15: "stairs",
            17: "walkable",
            30: "terrain",
        },

        "ADE20K": {

            3: "floor",
            6: "road",
            11: "sidewalk",
            53: "stairs",
        }
    }

    VEHICLE_CLASSES = ["car","motorcycle","bus","truck","bicycle"]
    PERSON_CLASS = "person"

    @classmethod
    def setup(cls):
        # Tạo folder output nếu chưa tồn tại
        os.makedirs(cls.OUT_DIR, exist_ok=True)

#### DEPTH LOADER

In [ ]:
def load_depth(path):
    try:
        with gzip.open(path,'rb') as f:
            # Đọc binary -> numpy array
            raw = np.frombuffer(f.read(),dtype=np.float16)

        h,w = int(raw[0]),int(raw[1])
        depth = raw[2:].reshape((h,w)).astype(np.float32)
        depth = cv2.resize(depth,(ThesisConfig.WIDTH,ThesisConfig.HEIGHT))
        valid = depth[depth > 0]
        # Chuẩn hóa depth
        if len(valid) > 0:
            # depth gần
            near = np.percentile(valid, 5)
            # depth xa
            far = np.percentile(valid, 95)
            # chuẩn hóa về 0-1
            depth = (depth - near) / (far - near + 1e-6)
            depth = np.clip(depth, 0, 1)
        return depth
    except Exception as e:
        print(f"[ERROR load_depth] {e}")
        return None

#### TEXT UTILS

In [ ]:
# ================= CLEAN TEXT =================
# làm sạch text warning trước khi đọc audio
def clean_text(msg):
    if not msg:
        return ""
    msg = msg.replace("nwuax", "")
    msg = msg.replace("nso", "")
    return msg.strip()

# ================= PRIORITY =================
def get_message_priority(msg):
    # Nguy hiểm cực cao
    if "Dừng lại, vật cản rất gần" in msg:
        return 5
    if "rất gần phía trước, dừng lại" in msg:
        return 5

    # cao
    if "Xe cộ rất đông, đứng yên" in msg:
        return 4
    if "Giao thông phức tạp, đi theo người bên cạnh" in msg:
        return 4
    if "Người chắn" in msg and "né sang" in msg:
        return 4

    # trung bình
    if "Có nhiều xe, hãy chú ý" in msg:
        return 3
    if "Có người" in msg and "đi chậm" in msg:
        return 3
    if "Vật cản" in msg and "gần, cẩn thận" in msg:
        return 3
    if "Có " in msg and " phía trước" in msg:
        return 3

    # thấp
    if "Phía trước bị chắn" in msg:
        return 2

    # rất thấp
    if "Có phương tiện giao thông" in msg:
        return 1
    if "có cầu thang" in msg or "mặt đường không phẳng" in msg:
        return 1
    if "Có thể đi tiếp" in msg:
        return 0

    # an toàn
    return 0

#### SEGMENTATION ENGINE

In [ ]:
def get_seg_label(seg_id):

    seg_id = int(seg_id)

    dataset_map = ThesisConfig.SEGMENTATION_ID2LABEL.get(
        ThesisConfig.DATASET_TYPE,
        {}
    )

    return dataset_map.get(
        seg_id,
        "unknown"
    )

def is_walkable_label(label):

    return (
        label
        in ThesisConfig.SEGMENTATION_SEMANTICS[
            "walkable"
        ]
    )

def get_walkable_ids():

    dataset_map = ThesisConfig.SEGMENTATION_ID2LABEL[
        ThesisConfig.DATASET_TYPE
    ]

    return np.array([
        seg_id
        for seg_id, label in dataset_map.items()
        if is_walkable_label(label)
    ])

WALKABLE_IDS = get_walkable_ids()

def is_stair_label(label):

    return (
        label
        in ThesisConfig.SEGMENTATION_SEMANTICS[
            "stairs"
        ]
    )

def is_on_walkable(seg, x1, y1, x2, y2):
    if seg is None:
        return True

    if len(seg.shape) == 3:
        seg = seg[:, :, 0]

    h, w = seg.shape
    cx = int((x1 + x2) / 2)
    cy = int(y2)

    cx = max(0, min(w - 1, cx))
    cy = max(0, min(h - 1, cy))

    crop = seg[
        max(0, cy-10):min(h, cy+10),
        max(0, cx-10):min(w, cx+10)
    ]

    if crop.size == 0:
        return False

    walkable_count = 0

    mask = np.isin(
        crop,
        WALKABLE_IDS
    )

    ratio = mask.mean()

    return ratio > 0.3

#### DEPTH ENGINE

In [ ]:
def get_stable_depth(depth, cx, cy):
        try:
            h, w = depth.shape
            patch = depth[max(0,cy-2):min(h,cy+3), max(0,cx-2):min(w,cx+3)]
            valid = patch[
                (patch > 0.02)
                & (patch < 0.98)
            ]
            if len(valid) > 0:
                return float(np.median(valid))
        except Exception as e:
            print(f"[ERROR stable_depth] {e}")
        return 5.0

#### NAVIGATION ENGINE

In [ ]:
def estimate_safe_direction(depth, seg):

    if depth is None or seg is None:
        return "straight", 0, None

    h, w = depth.shape

    # ==========================================
    # ROI
    # ==========================================

    roi_h1 = int(h * 0.35)
    roi_h2 = int(h * 0.95)

    depth_roi = depth[roi_h1:roi_h2]
    seg_roi = seg[roi_h1:roi_h2]

    # ==========================================
    # WALKABLE MASK
    # ==========================================

    walkable_mask = np.zeros_like(
        seg_roi,
        dtype=np.uint8
    )

    dataset_map = ThesisConfig.SEGMENTATION_ID2LABEL[
        ThesisConfig.DATASET_TYPE
    ]

    walkable_mask = np.isin(
        seg_roi,
        WALKABLE_IDS
    )

    # ==========================================
    # DEPTH FILTER
    # ==========================================

    valid_depth = (
        (depth_roi > 0.05)
        & (depth_roi < 0.95)
    )

    walkable_mask = (
        walkable_mask
        & valid_depth
    )

    # ==========================================
    # NO FREE SPACE
    # ==========================================

    if np.sum(walkable_mask) < 100:

        return "straight", 0, None

    # ==========================================
    # SAFE POINTS
    # ==========================================

    ys, xs = np.where(walkable_mask)

    if len(xs) == 0:

        return "straight", 0, None

    # ==========================================
    # 5 NAVIGATION REGIONS
    # ==========================================

    region_edges = np.linspace(
        0,
        depth_roi.shape[1],
        6
    ).astype(int)

    region_names = [
        "sharp_right",
        "slight_right",
        "straight",
        "slight_left",
        "sharp_left"
    ]

    best_score = -999
    best_region = "straight"
    best_x = depth_roi.shape[1] / 2

    for i in range(5):

        x1 = region_edges[i]
        x2 = region_edges[i+1]

        region_mask = walkable_mask[:, x1:x2]

        if np.sum(region_mask) < 50:
            continue

        region_depth = depth_roi[:, x1:x2]

        valid = region_depth[
            region_mask
        ]

        valid = valid[
            (valid > 0.05)
            & (valid < 0.95)
        ]

        if len(valid) < 30:
            continue

        # ===== SCORE =====

        free_score = np.mean(valid)

        walkable_score = (
            np.sum(region_mask)
            / region_mask.size
        )

        score = (
            free_score * 0.7
            + walkable_score * 0.3
        )

        if score > best_score:

            best_score = score

            best_region = region_names[i]

            best_x = (x1 + x2) / 2

    safe_x = best_x

    # convert ROI -> full image
    safe_x_global = safe_x

    # ==========================================
    # ANGLE
    # ==========================================

    center_x = w / 2

    offset = safe_x_global - center_x

    angle = (
        offset / center_x
    ) * 90

    angle = np.clip(angle, -90, 90)

    # ==========================================
    # DIRECTION
    # ==========================================

    if angle > 50:
        direction = "sharp_left"

    elif angle > 10:
        direction = "slight_left"

    elif angle < -50:
        direction = "sharp_right"

    elif angle < -10:
        direction = "slight_right"

    else:
        direction = "straight"

    debug_data = {
        "safe_x": int(safe_x),
        "roi_h1": roi_h1,
        "roi_h2": roi_h2
    }

    return direction, float(angle), debug_data

#### ANALYSIS

In [ ]:
with open(ThesisConfig.LABEL_COCO, "r", encoding="utf-8") as f:
        LABEL_MAP = json.load(f)
def analyze(frame, results, depth, seg, prev_frame=None):
    h, w = frame.shape[:2]

    # Initialize debug_nav to None
    debug_nav = None

    # resize seg nếu lệch size
    if depth is not None and seg is not None:
        if seg.shape[:2] != depth.shape:
            seg = cv2.resize(
                seg,
                (depth.shape[1], depth.shape[0]),
                interpolation=cv2.INTER_NEAREST
            )
    objects = []
    vehicle_count = 0
    person_nearby = False
    stair_detected = False
    small_obstacle = False

    blocked_front = False
    suggested_direction = ""

    safe_direction = "straight"
    safe_angle = 0

    all_potential_messages = [] # Moved initialization here

    # ==========================================
    # 1. QUÉT MẶT ĐƯỜNG
    # ==========================================
    if (
        ThesisConfig.USE_DEPTH
        and ThesisConfig.USE_SEGMENTATION
        and depth is not None
        and seg is not None
    ):
        roi_h1, roi_h2 = int(h * 0.6), int(h * 0.9)
        roi_w1, roi_w2 = int(w * 0.3), int(w * 0.7)

        depth_roi = depth[roi_h1:roi_h2, roi_w1:roi_w2]
        seg_roi = seg[roi_h1:roi_h2, roi_w1:roi_w2]

        if len(seg_roi.shape) == 3:
            seg_roi = seg_roi[:, :, 0]

        # ===== detect cầu thang =====
        stair_mask = np.zeros_like(
            seg_roi,
            dtype=bool
        )

        for seg_id in np.unique(seg_roi):

            label_name = get_seg_label(seg_id)

            if (
                label_name
                in ThesisConfig.SEGMENTATION_SEMANTICS["stairs"]
            ):

                stair_mask |= (seg_roi == seg_id)
        # Chỉ quét 1/2 phía dưới của ROI (phần sát chân người dùng)
        lower_half_h = int(seg_roi.shape[0] * 0.5)
        stair_mask_near = stair_mask[lower_half_h:, :]

        # Changed 0.05 to 0.10 for stricter stair detection
        if np.sum(stair_mask_near) > (stair_mask_near.size * 0.10):
            # Lấy depth của riêng vùng cầu thang
            stair_depths = depth_roi[lower_half_h:, :][stair_mask_near]
            # Nếu cầu thang thực sự ở gần (< 2.5m) mới báo
            if stair_depths.size > 0 and np.median(stair_depths) < 0.35:
                stair_detected = True

        # ===== depth ổn định =====
        valid_depth = depth_roi[
            (depth_roi > 0.02)
            & (depth_roi < 0.98)
        ]
        if len(valid_depth) > 0:
            min_dist = np.percentile(valid_depth, 5)
        else:
            min_dist = 5.0

        # print("min_dist:", min_dist)

        # ===== check vật cản =====
        if min_dist < 0.18:
            left_region = depth_roi[:, :int((roi_w2-roi_w1)*0.3)]
            right_region = depth_roi[:, int((roi_w2-roi_w1)*0.7):]

            left_valid = left_region[
                (left_region > 0.02)
                & (left_region < 0.98)
            ]
            right_valid = right_region[
                (right_region > 0.02)
                & (right_region < 0.98)
            ]

            if len(left_valid) > 0 and len(right_valid) > 0:
                left_c = np.percentile(left_valid, 20)
                right_c = np.percentile(right_valid, 20)

                diff = left_c - right_c
                # print("diff:", diff)

                if abs(diff) > 0.35:
                    blocked_front = True
                    suggested_direction = "trái" if diff > 0 else "phải"
                else:
                    pass

        # ===== check mặt đường gồ ghề =====

        walkable_mask = np.zeros_like(
            seg_roi,
            dtype=bool
        )

        dataset_map = ThesisConfig.SEGMENTATION_ID2LABEL[
            ThesisConfig.DATASET_TYPE
        ]

        for seg_id, label_name in dataset_map.items():

            if is_walkable_label(label_name):

                walkable_mask |= (seg_roi == seg_id)

        if np.sum(walkable_mask) > 100:

            road_depths = depth_roi[walkable_mask]

            # lọc depth nhiễu
            road_depths = road_depths[
                (road_depths > 0.05)
                & (road_depths < 0.95)
            ]

            if len(road_depths) > 100:

                depth_std = np.std(road_depths)

                depth_range = (
                    np.percentile(road_depths, 90)
                    - np.percentile(road_depths, 10)
                )

                # chỉ báo khi thật sự gồ ghề
                if (
                    depth_std > 0.7
                    and depth_range > 1.2
                ):

                    small_obstacle = True

                # ===== SAFE DIRECTION =====
                safe_direction, safe_angle, debug_nav = (
                    estimate_safe_direction(
                        depth,
                        seg
                    )
                )

    # =========================
    # 2. YOLO OBJECTS
    # =========================

    if results.boxes is not None:
        VALID_CLASSES = set(
            ThesisConfig.VEHICLE_CLASSES + [ThesisConfig.PERSON_CLASS]
        )
        track_ids = None

        if results.boxes.id is not None:
            track_ids = results.boxes.id.int().cpu().tolist()

        for idx, (box, cls) in enumerate(
            zip(results.boxes.xyxy, results.boxes.cls)
        ):
            # Initialize d at the start of the loop for each object
            d = 5.0
            # Initialize walkable status for the current object (default to True)
            current_object_is_walkable = True

            track_id = (
                track_ids[idx]
                if track_ids is not None
                else -1
            )
            label = results.names[int(cls)]

            # ===== FIX 1: lọc class rác =====
            if label not in VALID_CLASSES:
                continue
            x1, y1, x2, y2 = map(int, box)

            # ===== segmentation filter =====
            if ThesisConfig.USE_SEGMENTATION:

                # Calculate walkable status for the current object
                current_object_is_walkable = is_on_walkable(
                    seg,
                    x1,
                    y1,
                    x2,
                    y2
                )

                # ===== PERSON =====
                if label == ThesisConfig.PERSON_CLASS:

                    if not current_object_is_walkable:
                        continue

                # ===== VEHICLE =====


            cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)

            if (
                ThesisConfig.USE_DEPTH
                and depth is not None
            ):
                d = get_stable_depth(
                    depth,
                    cx,
                    cy
                )
            # ===== VEHICLE WALKABLE FILTER =====

            if (
                label in ThesisConfig.VEHICLE_CLASSES
                and not current_object_is_walkable
            ):

                if d > 0.18:
                    continue

            if label == ThesisConfig.PERSON_CLASS:
                if abs(cx - w/2) > w * 0.50:  # w * 0.35
                    continue
            area = (x2 - x1) * (y2 - y1) / (w * h)
            if label != ThesisConfig.PERSON_CLASS and cy < h * 0.22:
                continue

            if label != ThesisConfig.PERSON_CLASS and area < 0.002:
                continue


            # lọc nhiễu người
            # chỉ dùng khi có depth
            if (
                ThesisConfig.USE_DEPTH
                and label == ThesisConfig.PERSON_CLASS
            ):
                if d > 0.95 or d <= 0.02:
                    continue

            if (
                label in ThesisConfig.VEHICLE_CLASSES
                and current_object_is_walkable
            ):
                vehicle_count += 1

            if label == ThesisConfig.PERSON_CLASS:

                if ThesisConfig.USE_DEPTH:

                    if d < 0.55 and area > 0.003:
                        person_nearby = True

                else:

                    if area > 0.003:
                        person_nearby = True

            objects.append({
                "track_id": track_id,
                "label": label,
                "depth": d,
                "area": area,
                "cx": cx,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "is_walkable": current_object_is_walkable,
            })


    # =========================
    # 3. RISK
    # =========================
    total_risk, max_risk, most_dangerous = 0, 0, None

    for obj in objects:
        dist_score = max(0, 1 - obj["depth"])
        size_score = min(obj["area"] * 5, 1.0)
        center_score = 1 - abs(obj["cx"] - w/2) / (w/2)
        if obj["label"] in ThesisConfig.VEHICLE_CLASSES:
            type_score = 2.0

        elif obj["label"] == ThesisConfig.PERSON_CLASS:
            type_score = 3.5

        else:
            type_score = 1.0
        risk = (dist_score*0.5 + size_score*0.3 + center_score*0.2) * type_score

        total_risk += risk
        if risk > max_risk:
            max_risk = risk
            most_dangerous = obj
    objects = sorted(objects, key = lambda o: (
        max(0, 1 - o["depth"]) + o["area"] * 5
    ), reverse=True)

    top_objects = objects[:3]

    nearest_vehicle_depth = 999

    for obj in objects:

        if obj["label"] in ThesisConfig.VEHICLE_CLASSES:

            nearest_vehicle_depth = min(
                nearest_vehicle_depth,
                obj["depth"]
            )

    # ==========================================
    # 4. OUTPUT - Collect all potential messages with priorities and return the highest one
    # ==========================================

    # all_potential_messages = [] # Original initialization location

    # 1. Immediate/Critical Obstacle (Priority 5)
    if blocked_front and min_dist < 1.2:
        all_potential_messages.append((5, f"Dừng lại, vật cản rất gần, hãy né sang {suggested_direction}"))

    # 2. Check for very close/dangerous single vehicle
    if (
        most_dangerous
        and most_dangerous['label'] in ThesisConfig.VEHICLE_CLASSES
    ):

        vehicle_on_walkable = True

        if (
            ThesisConfig.USE_SEGMENTATION
            and seg is not None
        ):
            # Use the 'is_walkable' status stored in the object directly
            vehicle_on_walkable = most_dangerous["is_walkable"]

        # chỉ stop nếu xe ảnh hưởng đường đi
        if (
            vehicle_on_walkable
            and most_dangerous['depth'] < 0.15
        ):

            label_vi = LABEL_MAP.get(
                most_dangerous['label'],
                most_dangerous['label']
            )

            pos = (
                "bên trái"
                if most_dangerous['cx'] < w/2 - 100
                else (
                    "bên phải"
                    if most_dangerous['cx'] > w/2 + 100
                    else "phía trước"
                )
            )

            all_potential_messages.append(
                (5, f"{label_vi} rất gần {pos}, dừng lại")
            )
    # 4. High-risk multiple vehicles or traffic with person (Priority 4) - Moved before person blocking path for priority

    # ===== cực đông =====
    if (
        vehicle_count >= 4
        and max_risk > 2.5
        and nearest_vehicle_depth < 0.25
    ):

        all_potential_messages.append(
            (5, "Xe cộ rất đông, đứng yên")
        )

    # ===== giao thông phức tạp =====
    elif (
        vehicle_count >= 1
        and person_nearby
        and (
            not ThesisConfig.USE_DEPTH
            or nearest_vehicle_depth < 0.35
        )
    ):

        all_potential_messages.append(
            (4, "Giao thông phức tạp, đi theo người bên cạnh")
        )

    # ===== nhiều xe =====
    elif vehicle_count >= 1:

        all_potential_messages.append(
            (3, "Có nhiều xe, hãy chú ý")
        )

    # ===== ít xe nhưng gần =====
    elif (
        vehicle_count >= 2
        and max_risk > 1.0
    ):

        all_potential_messages.append(
            (3, "Có nhiều xe, hãy chú ý")
        )

    # 3. Person blocking path (Priority 4)
    if (
        most_dangerous
        and most_dangerous["label"] == ThesisConfig.PERSON_CLASS
        and most_dangerous["depth"] < 0.45
        and abs(most_dangerous["cx"] - w/2) < w * 0.30
        and most_dangerous["area"] > 0.015
    ):
        side = "phải" if most_dangerous['cx'] < w/2 else "trái"
        pos = "bên trái" if most_dangerous['cx'] < w/2 - 100 else ("bên phải" if most_dangerous['cx'] > w/2 + 100 else "phía trước")
        all_potential_messages.append((2, f"Người chắn {pos}, né sang {side}"))

    # 6. General single vehicle warning (Priority 3)
    elif vehicle_count == 1 and most_dangerous and most_dangerous['label'] in ThesisConfig.VEHICLE_CLASSES:
        if most_dangerous['depth'] < 0.55: # General single vehicle, not critically close
            all_potential_messages.append((3, f"Có {LABEL_MAP.get(most_dangerous['label'], most_dangerous['label'])} phía trước"))

    # 7. General object warnings (Priority 3) - for other objects or persons
    if most_dangerous:
        label_vi = LABEL_MAP.get(most_dangerous["label"], most_dangerous["label"])
        d = most_dangerous["depth"]
        cx = most_dangerous["cx"]
        pos = "bên trái" if cx < w/2 - 100 else ("bên phải" if cx > w/2 + 100 else "phía trước")

        if most_dangerous["label"] == ThesisConfig.PERSON_CLASS:

            if ThesisConfig.USE_DEPTH:

                if (
                    d < 0.7
                    and abs(cx - w/2) < 450
                ):
                    all_potential_messages.append(
                        (3, f"Có người {pos}, đi chậm")
                    )

            else:

                all_potential_messages.append(
                    (3, f"Có người {pos}, đi chậm")
                )
        elif d < 0.25 and most_dangerous['label'] not in ThesisConfig.VEHICLE_CLASSES and most_dangerous['label'] != ThesisConfig.PERSON_CLASS:
            all_potential_messages.append((3, f"Vật cản {pos} gần, cẩn thận"))

    # 8. Farther Blocked Path (Priority 2)
    if blocked_front and min_dist >= 1.2:
        all_potential_messages.append((2, f"Phía trước bị chắn, đi lệch sang {suggested_direction}"))

    # 9. Surface Conditions (Priority 1)
    surface_warnings = []
    if stair_detected:
        surface_warnings.append("có cầu thang")
    if small_obstacle:
        surface_warnings.append("mặt đường không phẳng")
    # chỉ báo nếu KHÔNG có object nguy hiểm
    if (
        surface_warnings
        and vehicle_count == 0
        and not person_nearby
    ):

        all_potential_messages.append(
            (
                1,
                " và ".join(surface_warnings)
            )
        )
    # Prioritize and return the highest priority message
    if all_potential_messages:

        all_potential_messages.sort(
            key=lambda x: x[0],
            reverse=True
        )

        danger_type = None

        if most_dangerous:
            danger_type = most_dangerous["label"]

        final_msg = all_potential_messages[0][1]

        return {
            "message": final_msg,
            "danger_type": danger_type,
            "direction": safe_direction,
            "angle": safe_angle,
            "debug_nav": debug_nav
        }

    return {
        "message": "",
        "danger_type": None,
        "direction": safe_direction,
        "angle": safe_angle,
        "debug_nav": debug_nav
    }

#### MAIN

In [ ]:
def phase1_analyze():
    start_time = time.time()
    ThesisConfig.setup()
    model = YOLO("yolo11n.pt")

    if torch.cuda.is_available():

        model.to("cuda")

    files = sorted(os.listdir(ThesisConfig.IMG_DIR))
    messages = []
    predictions = {}
    SAFE_MSG = "Có thể đi tiếp"

    prev_msg = SAFE_MSG
    hold_count = 0
    traffic_memory = 0
    last_warning_msg = ""

    last_emit_frame = -999
    #COOLDOWN = 90 #SANPO
    COOLDOWN = 150
    cached_results = None
    cached_direction = "straight"
    cached_angle = 0

    stable_direction = "straight"
    direction_hold = 0

    for i, f in enumerate(files):
        #print(f"[{i+1}/{len(files)}] {f}")

        frame = cv2.imread(os.path.join(ThesisConfig.IMG_DIR,f))
        if frame is None:
            msg = SAFE_MSG
        else:
            frame = cv2.resize(frame,(ThesisConfig.WIDTH,ThesisConfig.HEIGHT))

            name = os.path.splitext(f)[0]
            depth = load_depth(os.path.join(ThesisConfig.DEPTH_DIR,name+".float16.gz"))
            seg = cv2.imread(os.path.join(ThesisConfig.SEG_DIR,f),0)

            #SKIP = 10 #SANPO
            SKIP = 20

            if i % SKIP == 0 or cached_results is None:

                with torch.no_grad():

                    cached_results = model(
                        frame,
                        imgsz=640,
                        conf=0.35,
                        verbose=False,
                        device=0 if torch.cuda.is_available() else "cpu",
                        half=torch.cuda.is_available()
                    )[0]

            results = cached_results

            # ===== planner không cần chạy every frame =====
            if i % SKIP == 0:

                analysis = analyze(
                    frame,
                    results,
                    depth,
                    seg
                )

                new_direction = analysis.get(
                    "direction",
                    "straight"
                )

                if new_direction == stable_direction:

                    direction_hold = 0

                else:

                    direction_hold += 1

                    if direction_hold >= 5:

                        stable_direction = new_direction
                        direction_hold = 0

                cached_direction = stable_direction

                new_angle = analysis.get(
                    "angle",
                    0
                )

                cached_angle = (
                    0.7 * cached_angle
                    + 0.3 * new_angle
                )

            else:

                analysis = {
                    "message": prev_msg,
                    "danger_type": None,
                    "direction": cached_direction,
                    "angle": cached_angle,
                    "debug_nav": None
                }

            msg = clean_text(
                analysis["message"]
            )

            # ==========================================
            # SAVE PREDICTION
            # ==========================================

            predictions[name] = {

                "direction": cached_direction,
                "angle": float(
                    cached_angle
                )
            }
            if not msg.strip():
                msg = SAFE_MSG

            #print(f"  [DEBUG] VisionEngine Raw Msg: '{msg}' (P: {get_message_priority(msg)})")

            danger_type = analysis.get(
                "danger_type",
                None
            )

        # ===== SMART SMOOTH =====
        #MIN_HOLD = 45 # SANPO
        MIN_HOLD = 60
        candidate = msg
        #print(f"  [DEBUG] Initial Candidate: '{candidate}' (P: {get_message_priority(candidate)}) ")
        if get_message_priority(candidate) >= 2:
            last_warning_msg = candidate

        # ===== nhớ trạng thái giao thông =====
        if (
            "xe" in candidate.lower()
            or "giao thông" in candidate.lower()
            or "người" in candidate.lower()
        ):

            traffic_memory = 15

        else:

            traffic_memory = max(0, traffic_memory - 1)
        # ===== KHÔNG CHO SAFE XUẤT HIỆN QUÁ NHANH =====
        if (
            candidate == ""
            or candidate == SAFE_MSG
        ):
            # ===== nếu vừa có traffic =====
            if traffic_memory > 0:

                # chỉ giữ warning traffic
                if (
                    "xe" in prev_msg.lower()
                    or "giao thông" in prev_msg.lower()
                    or "người" in prev_msg.lower()
                ):

                    candidate = prev_msg
            # ===== giữ warning cũ =====
            elif (
                prev_msg != SAFE_MSG
                and hold_count < MIN_HOLD * 1.3
            ):
                candidate = prev_msg
                #print(f"  [DEBUG]   -> Holding Prev Msg (warning): '{candidate}'")

            else:
                candidate = SAFE_MSG
                #print(f"  [DEBUG]   -> Candidate set to SAFE_MSG: '{candidate}'")


        candidate_p = get_message_priority(candidate)
        current_p = get_message_priority(prev_msg)

        #print(f"  [DEBUG] After smoothing attempt: Candidate: '{candidate}' (P: {candidate_p}), Prev Msg: '{prev_msg}' (P: {current_p}), Hold Count: {hold_count})")

        if candidate == prev_msg:
            hold_count += 1
        elif candidate_p > current_p:
            if hold_count >= MIN_HOLD * 0.6:
                #print(f"  [DEBUG]   -> New candidate higher priority, sufficient hold: Adopting candidate.")
                prev_msg = candidate
                hold_count = 1
            else:
                #print(f"  [DEBUG]   -> New candidate higher priority, insufficient hold: Holding previous.")
                hold_count += 1
        elif candidate_p == current_p:
            if current_p >= 2:
                if hold_count < MIN_HOLD * 2:
                    #print(f"  [DEBUG]   -> Same priority, warning, holding previous.")
                    hold_count += 1
                else:
                    #print(f"  [DEBUG]   -> Same priority, warning, hold limit reached: Adopting candidate.")
                    prev_msg = candidate
                    hold_count = 1
            else:
                if hold_count < MIN_HOLD:
                    #print(f"  [DEBUG]   -> Same priority, not warning, holding previous.")
                    hold_count += 1
                else:
                    #print(f"  [DEBUG]   -> Same priority, not warning, hold limit reached: Adopting candidate.")
                    prev_msg = candidate
                    hold_count = 1
        else: # candidate_p < current_p
            if hold_count >= MIN_HOLD * 2:
                #print(f"  [DEBUG]   -> New candidate lower priority, sufficient hold: Adopting candidate.")
                prev_msg = candidate
                hold_count = 1
            else:
                #print(f"  [DEBUG]   -> New candidate lower priority, insufficient hold: Holding previous.")
                hold_count += 1

        # ===== FINAL MESSAGE =====
        final_msg = prev_msg

        # ===== chống nhảy warning liên tục =====
        if (
            candidate_p == current_p
            and candidate != prev_msg
            and current_p >= 3
        ):
            # giữ warning cũ thêm một lúc
            if hold_count < MIN_HOLD * 3:
                candidate = prev_msg
                #print(f"  [DEBUG]   -> Anti-jump: Same high priority, candidate different from prev_msg, holding prev_msg: '{candidate}'")


        # nếu warning mới nguy hiểm hơn
        if candidate_p > current_p:
            #print(f"  [DEBUG]   -> Anti-jump: Candidate higher priority: Adopting candidate.")
            final_msg = candidate

        # nếu cùng priority
        elif candidate_p == current_p:

            # chỉ đổi nếu hold đủ lâu
            if hold_count >= MIN_HOLD:
                #print(f"  [DEBUG]   -> Anti-jump: Same priority, sufficient hold: Adopting candidate.")
                final_msg = candidate
            else:
                #print(f"  [DEBUG]   -> Anti-jump: Same priority, insufficient hold: Sticking with '{final_msg}'.")
                pass

        # nếu thấp hơn
        else: # candidate_p < current_p

            # giữ warning cũ
            if hold_count >= MIN_HOLD * 2:
                #print(f"  [DEBUG]   -> Anti-jump: Candidate lower priority, sufficient hold: Adopting candidate.")
                final_msg = candidate
            else:
                #print(f"  [DEBUG]   -> Anti-jump: Candidate lower priority, insufficient hold: Sticking with '{final_msg}'.")
                pass

        direction_vi = {

            "sharp_left": "rẽ trái mạnh",

            "slight_left": "lệch trái nhẹ",

            "straight": "đi thẳng",

            "slight_right": "lệch phải nhẹ",

            "sharp_right": "rẽ phải mạnh"
        }

        nav_text = direction_vi[
            cached_direction
        ]

        if final_msg == SAFE_MSG:

            final_msg_with_nav = (
                SAFE_MSG
                + ", "
                + nav_text
            )

        else:

            final_msg_with_nav = (
                final_msg
                + ", "
                + nav_text
            )

        # append event
        if (
            len(messages) == 0
            or (
                final_msg_with_nav != messages[-1]["message"]
                and (i - last_emit_frame > COOLDOWN)
            )

        ):

            messages.append({
                "frame": i,
                "message": final_msg_with_nav,
                "danger_type": danger_type,
                "direction": cached_direction,

                "angle": float(cached_angle),
            })

            last_emit_frame = i
    # ===== ADD DURATION =====
    for idx in range(len(messages)):
        start = messages[idx]["frame"]

        if idx < len(messages) - 1:
            end = messages[idx+1]["frame"]
        else:
            end = len(files) + 120

        messages[idx]["duration_frames"] = end - start
    #out = os.path.join(ThesisConfig.OUT_DIR,"messages_full5.json")
    out = os.path.join(ThesisConfig.OUT_DIR,"messages_full_my_data.json")
    pred_out = os.path.join(
        ThesisConfig.OUT_DIR,
        #"predictions5.json"
        "predictions_my_data.json"
    )
    with open(out,"w",encoding="utf-8") as f:
        json.dump(messages,f,ensure_ascii=False,indent=2)
    with open(
        pred_out,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            predictions,
            f,
            ensure_ascii=False,
            indent=2
        )

    print("DONE:", out)
    print(
        "Runtime:",
        round(time.time() - start_time, 2),
        "seconds"
    )

phase1_analyze()

DONE: /content/drive/MyDrive/KLTN/output/messages_full_my_data.json
Runtime: 1286.73 seconds


#### PHASE 2

In [ ]:
OUT_DIR = "/content/drive/MyDrive/KLTN/output"
IMG_DIR = "/content/drive/MyDrive/KLTN/my_data_ver2/video_frames"
#IMG_DIR = "/content/drive/MyDrive/KLTN/dataset/data5/left/video_frames"
WIDTH, HEIGHT = 1242, 736
#FPS = 10 # SANPO
FPS = 30
model = YOLO("yolo11n.pt")
FONT_PATH = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"

# ================= NORMALIZE MESSAGE =================
def normalize_msg(msg_str):
    if not msg_str:
        return "Có thể đi tiếp"

    msg_str = msg_str.strip()

    # fix lỗi chính tả phổ biến
    msg_str = msg_str.replace("phia", "phía")
    msg_str = msg_str.replace("trai", "trái")
    msg_str = msg_str.replace("phai", "phải")

    return msg_str

def get_message_priority(msg):
    # Nguy hiểm cực cao
    if "Dừng lại, vật cản rất gần" in msg:
        return 5
    if "rất gần phía trước, dừng lại" in msg:
        return 5

    # Nguy hiểm cao
    if "Xe cộ rất đông, đứng yên" in msg:
        return 4
    if "Giao thông phức tạp, đi theo người bên cạnh" in msg:
        return 4
    if "Người chắn" in msg and "né sang" in msg:
        return 4

    # Cảnh báo trung bình
    if "Có nhiều xe, hãy chú ý" in msg:
        return 3
    if "Có người" in msg and "đi chậm" in msg:
        return 3
    if "Vật cản" in msg and "gần, cẩn thận" in msg:
        return 3
    if "Có " in msg and " phía trước" in msg:
        return 3

    if "Phía trước bị chắn" in msg:
        return 2

    if "Có phương tiện giao thông" in msg:
        return 1
    if "có cầu thang" in msg or "mặt đường không phẳng" in msg:
        return 1

    # An toàn
    return 0
# ================= AUDIO =================
def generate_audio(messages):
    print("Generating audio (frame-based)...")

    frame_ms = int(1000 / FPS)
    final_audio = AudioSegment.empty()

    cache = {}

    for idx, msg_data in enumerate(messages):
        msg_text = normalize_msg(
            msg_data.get('message', '')
        )

        if not msg_text.strip():
            msg_text = "Có thể đi tiếp"
        priority = get_message_priority(msg_text)

        duration = msg_data["duration_frames"] * frame_ms

        # ===== SAFE MESSAGE =====
        if msg_text == "Có thể đi tiếp":

            prev_msg = ""

            if idx > 0:
                prev_msg = messages[idx - 1]["message"]

            # chỉ đọc SAFE nếu trước đó là warning
            if get_message_priority(prev_msg) >= 3:

                safe_text = "Có thể đi tiếp"

                if safe_text not in cache:

                    path = f"/content/audio_safe.mp3"

                    gTTS(
                        text=safe_text,
                        lang='vi'
                    ).save(path)

                    cache[safe_text] = AudioSegment.from_mp3(path)

                audio_seg_to_process = cache[safe_text]

                audio_len = len(audio_seg_to_process)

                if audio_len < duration:

                    silence = AudioSegment.silent(
                        duration=duration - audio_len
                    )

                    final_segment = (
                        audio_seg_to_process + silence
                    )

                else:

                    final_segment = audio_seg_to_process

                final_audio += final_segment

            else:

                final_audio += AudioSegment.silent(
                    duration=duration
                )

            continue

        if msg_text not in cache:
            path = f"/content/audio_{len(cache)}.mp3"

            tts_text = msg_text.replace(
                ",",
                ". "
            )
            gTTS(text=msg_text, lang='vi', slow=True).save(path)
            cache[msg_text] = AudioSegment.from_mp3(path)

        audio_seg_to_process = cache[msg_text]

        duration = msg_data["duration_frames"] * frame_ms # Use msg_data (the dictionary) for duration

        # ===== giữ tốc độ đọc tự nhiên =====
        audio_len = len(audio_seg_to_process)

        # nếu audio ngắn hơn duration → thêm silence
        if audio_len < duration:
            silence = AudioSegment.silent(duration=duration - audio_len)
            final_segment = audio_seg_to_process + silence

        # nếu audio dài hơn duration → KHÔNG tua nhanh
        else:
            final_segment = (
                audio_seg_to_process
            )

        final_audio += final_segment

    out = "/content/audio.mp3"
    final_audio.export(out, format="mp3")

    print("Audio saved:", out)
    return out

# ================= VIDEO =================
def render_video(messages):
    print("Rendering video...")
    font = ImageFont.truetype(FONT_PATH, 32)
    img_files = sorted(os.listdir(IMG_DIR))
    out_path = "/content/video.mp4"

    vw = cv2.VideoWriter(
        out_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        FPS,
        (WIDTH, HEIGHT)
    )

    last_yolo_render_results = None # Initialize to store YOLO results for reuse
    last_yolo_render_frame_idx = -1 # Initialize to store the frame index of the last YOLO detection

    current_frame_idx = 0
    # frame_ms = int(1000 / FPS) # This variable is not used in render_video
    for msg_data in messages:
        msg_text = normalize_msg(
            msg_data.get('message', '')
        )
        priority = get_message_priority(msg_text)

        # màu theo mức độ nguy hiểm
        if priority >= 5:
            color = (0, 0, 255)   # đỏ
        elif priority >= 4:
            color = (0, 165, 255) # cam
        elif priority >= 3:
            color = (0, 255, 255) # vàng
        else:
            color = (0, 255, 0)   # xanh

        frame_duration = msg_data['duration_frames']

        for _ in range(frame_duration):
            if current_frame_idx >= len(img_files):
                break

            f = img_files[current_frame_idx]
            frame = cv2.imread(os.path.join(IMG_DIR, f))
            frame = cv2.resize(frame, (WIDTH, HEIGHT))

            direction = msg_data.get(
                "direction",
                "straight"
            )

            direction_angle = {

                "sharp_left": -60,

                "slight_left": -30,

                "straight": 0,

                "slight_right": 30,

                "sharp_right": 60

            }

            angle = direction_angle[
                direction
            ]

            center_x  = WIDTH // 2
            center_y = HEIGHT - 120

            arrow_len = 180

            end_x = int(center_x + arrow_len * np.sin(np.radians(angle)))
            end_y = int(center_y - arrow_len * np.cos(np.radians(angle)))
            cv2.arrowedLine(
                frame,
                (center_x, center_y),
                (end_x, end_y),
                (0, 255, 0),
                8,
                tipLength=0.25
            )
            cv2.circle(
                frame,
                (center_x, center_y),
                18,
                (255, 255, 255),
                -1
            )

            # ==========================================
            # DRAW ANGLE
            # ==========================================

            direction = msg_data.get("direction", "straight")

            cv2.putText(frame,
                        f"Direction: {direction}",
                        (40, HEIGHT - 80),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1,
                        (255, 255, 255),
                        2
            )

            # ===== YOLO DETECT REALTIME (Optimized) =====
            # Assuming ThesisConfig is available from a previously executed cell.
            # If not, this will raise a NameError.
            results_to_draw = None
            # Only perform YOLO detection if it's the first frame or enough frames have skipped
            if current_frame_idx == 0 or (current_frame_idx - last_yolo_render_frame_idx) >= ThesisConfig.YOLO_RENDER_SKIP_INTERVAL:
                results_to_draw = model(
                    frame,
                    imgsz=640,
                    conf=ThesisConfig.YOLO_CONF,
                    verbose=False
                )[0]
                last_yolo_render_results = results_to_draw
                last_yolo_render_frame_idx = current_frame_idx
                #print(f"YOLO DETECT (NEW) on frame {current_frame_idx}: {len(results_to_draw.boxes)}")
            else:
                # Reuse last YOLO detection results
                results_to_draw = last_yolo_render_results
                #print(f"YOLO DETECT (REUSED) on frame {current_frame_idx} (from frame {last_yolo_render_frame_idx}): {len(results_to_draw.boxes) if results_to_draw and results_to_draw.boxes is not None else 0}")

            # Fallback for drawing if no results could be obtained or reused (e.g., first frame and skip is 0, or an error)
            if results_to_draw is None:
                 results_to_draw = model(
                    frame,
                    imgsz=640,
                    conf=ThesisConfig.YOLO_CONF,
                    verbose=False
                )[0]
                 last_yolo_render_results = results_to_draw
                 last_yolo_render_frame_idx = current_frame_idx
                 #print(f"YOLO DETECT (FALLBACK NEW) on frame {current_frame_idx}: {len(results_to_draw.boxes)}")

            # ===== DRAW REALTIME BBOX =====

            danger_type = msg_data.get(
                "danger_type",
                None
            )
            if danger_type is None:

                lower_msg = msg_text.lower()

                if "người" in lower_msg:
                    danger_type = "person"

                elif "xe" in lower_msg:
                    danger_type = "car"

            # Ensure results_to_draw and its boxes exist before iterating
            if results_to_draw and results_to_draw.boxes is not None:
                for box, cls, conf in zip(
                    results_to_draw.boxes.xyxy,
                    results_to_draw.boxes.cls,
                    results_to_draw.boxes.conf
                ):

                    label = results_to_draw.names[int(cls)]

                    confidence = float(conf)

                    # bỏ detect yếu
                    x1, y1, x2, y2 = map(int, box)

                    # ===== OBJECT INFO =====
                    cx = (x1 + x2) // 2
                    cy = (y1 + y2) // 2

                    bbox_area = (
                        (x2 - x1)
                        * (y2 - y1)
                    ) / (WIDTH * HEIGHT)

                    # ===== VALID CLASSES =====
                    VALID_CLASSES = {
                        "person",
                        "car",
                        "motorcycle",
                        "bus",
                        "truck",
                        "bicycle"
                    }

                    if label not in VALID_CLASSES:
                        continue

                    # ===== CONFIDENCE =====
                    if confidence < 0.45:
                        continue

                    # ===== OBJECT TOO SMALL =====
                    if bbox_area < 0.006:
                        continue

                    # ===== OBJECT TOO FAR =====
                    if cy < HEIGHT * 0.32:
                        continue

                    # ===== PERSON TOO SIDE =====
                    if label == "person":

                        if abs(cx - WIDTH / 2) > WIDTH * 0.35:
                            continue

                    # ===== VEHICLE TOO SIDE =====
                    else:

                        if abs(cx - WIDTH / 2) > WIDTH * 0.45:
                            continue

                    # ===== ONLY SHOW DANGER OBJECT =====
                    if danger_type is not None:

                        if label != danger_type:
                            continue

                    # ===== DRAW STYLE =====
                    box_color = (0, 0, 255)
                    thickness = 4

                    cv2.rectangle(
                        frame,
                        (x1, y1),
                        (x2, y2),
                        box_color,
                        thickness
                    )

                    cv2.putText(
                        frame,
                        f"{label} {confidence:.2f}",
                        (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8,
                        box_color,
                        2
                    )

            cv2.rectangle(frame, (0, 0), (WIDTH, 70), (0, 0, 0), -1)

            # ===== OpenCV -> PIL =====
            pil_img = Image.fromarray(
                cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            )

            draw = ImageDraw.Draw(pil_img)

            # đổi BGR -> RGB
            pil_color = (color[2], color[1], color[0])

            draw.text(
                (20, 20),
                msg_text,
                font=font,
                fill=pil_color
            )

            # ===== PIL -> OpenCV =====
            frame = cv2.cvtColor(
                np.array(pil_img),
                cv2.COLOR_RGB2BGR
            )
            vw.write(frame)
            current_frame_idx += 1
        if current_frame_idx >= len(img_files):
            break

    vw.release()
    print("Video saved:", out_path)
    return out_path


# ================= MERGE =================
def merge(video, audio):
    print("Merging video + audio...")

    out = os.path.join(OUT_DIR, "final_full_my_data.mp4")
    #out = os.path.join(OUT_DIR, "final_full5.mp4")
    os.system(
        f"ffmpeg -y -i {video} -i {audio} -c:v copy -c:a aac {out}"
    )

    print("Final video:", out)
    return out


# ================= MAIN =================
def phase2_render():
    msg_path = os.path.join(OUT_DIR, "messages_full_my_data.json")
    #msg_path = os.path.join(OUT_DIR, "messages_full5.json")
    if not os.path.exists(msg_path):
        print("Thiếu messages.json → chạy Phase 1 trước")
        return

    messages = json.load(open(msg_path, encoding="utf-8"))

    print("Total messages/segments:", len(messages))

    video = render_video(messages)
    audio = generate_audio(messages)

    final = merge(video, audio)

    print("DONE PHASE 2:", final)

In [ ]:
import os

!apt-get update -qq
!apt-get install -qq -y fonts-dejavu-core

phase2_render()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Total messages/segments: 9
Rendering video...
Video saved: /content/video.mp4
Generating audio (frame-based)...
Audio saved: /content/audio.mp3
Merging video + audio...
Final video: /content/drive/MyDrive/KLTN/output/final_full_my_data.mp4
DONE PHASE 2: /content/drive/MyDrive/KLTN/output/final_full_my_data.mp4


#### Evaluation

In [ ]:
import json
import numpy as np

GT_PATH = "/content/drive/MyDrive/KLTN/ground_truth_my_data.json"
#GT_PATH = "/content/drive/MyDrive/KLTN/ground_truth5.json"

PRED_PATH = "/content/drive/MyDrive/KLTN/output/predictions_my_data.json"
#PRED_PATH = "/content/drive/MyDrive/KLTN/output/predictions5.json"

with open(GT_PATH, "r") as f:
    gt = json.load(f)

with open(PRED_PATH, "r") as f:
    pred = json.load(f)

# ==========================================
# DEBUG KEYS
# ==========================================

print("="*50)

print("GT sample keys:")
print(list(gt.keys())[:5])

print()

print("Prediction sample keys:")
print(list(pred.keys())[:5])

print("="*50)

# ==========================================
# DIRECTION ORDER
# ==========================================

direction_order = {

    "sharp_left": 0,

    "slight_left": 1,

    "straight": 2,

    "slight_right": 3,

    "sharp_right": 4
}

# ==========================================
# METRICS STORAGE
# ==========================================

relaxed_correct = 0

critical_errors = 0

total = 0

# ==========================================
# LOOP
# ==========================================

for frame_id_raw in gt:

    # ==========================================
    # REMOVE .png IF EXISTS
    # ==========================================

    frame_id = frame_id_raw.replace(
        ".png",
        ""
    )

    # ==========================================
    # SKIP IF PRED NOT FOUND
    # ==========================================

    if frame_id not in pred:
        continue

    # ==========================================
    # LOAD VALUES
    # ==========================================

    gt_dir = gt[frame_id_raw]["direction"]

    pred_dir = pred[frame_id]["direction"]


    # ==========================================
    # RELAXED DIRECTION ACCURACY
    # ==========================================

    gt_idx = direction_order[gt_dir]

    pred_idx = direction_order[pred_dir]

    if abs(gt_idx - pred_idx) <= 1:

        relaxed_correct += 1

    # ==========================================
    # CRITICAL ERROR
    # ==========================================

    if abs(gt_idx - pred_idx) >= 3:

        critical_errors += 1

    total += 1

# ==========================================
# CHECK TOTAL
# ==========================================

if total == 0:

    print("ERROR: No matching frames found!")

    exit()

# ==========================================
# FINAL METRICS
# ==========================================

relaxed_acc = (
    relaxed_correct / total
) * 100

critical_error_rate = (
    critical_errors / total
) * 100

# ==========================================
# PRINT RESULTS
# ==========================================

print("="*60)

print("TOTAL EVALUATED FRAMES:", total)

print("="*60)

print(
    "Relaxed Direction Accuracy:",
    round(relaxed_acc, 2),
    "%"
)

print(
    "Critical Error Rate:",
    round(critical_error_rate, 2),
    "%"
)

print("="*60)

GT sample keys:
['000000', '000020', '000040', '000060', '000080']

Prediction sample keys:
['000000', '000001', '000002', '000003', '000004']
TOTAL EVALUATED FRAMES: 106
Relaxed Direction Accuracy: 76.42 %
Critical Error Rate: 1.89 %


#### Tạo ground

In [ ]:
import json
import numpy as np

# ==========================================
# INPUT
# ==========================================

CVAT_JSON = "/content/mydata1.json"

# ==========================================
# OUTPUT
# ==========================================

SAVE_PATH = "/content/drive/MyDrive/KLTN/ground_truth_my_data_ver2.json"

# ==========================================
# IMAGE SIZE
# ==========================================

WIDTH = 576
#WIDTH = 2208 #SANPO
HEIGHT = 1024
#HEIGHT = 1242 #SANPO
CENTER_X = WIDTH / 2

# ==========================================
# LOAD CVAT JSON
# ==========================================

with open(CVAT_JSON, "r") as f:
    data = json.load(f)

# ==========================================
# IMAGE ID -> FILE NAME
# ==========================================

image_map = {}

for img in data["images"]:

    image_map[img["id"]] = img["file_name"]

# ==========================================
# OUTPUT GT
# ==========================================

gt = {}

# ==========================================
# LOOP ANNOTATIONS
# ==========================================

for ann in data["annotations"]:

    image_id = ann["image_id"]

    file_name = image_map[image_id]

    frame_id = file_name.replace(
        ".png",
        ""
    )

    # ==========================================
    # POLYGON
    # ==========================================

    poly = ann["segmentation"][0]

    xs = poly[0::2]
    ys = poly[1::2]

    # ==========================================
    # CENTROID
    # ==========================================

    cx = np.mean(xs)

    cy = np.mean(ys)

    # ==========================================
    # ANGLE
    # ==========================================

    offset = cx - CENTER_X

    angle = (
        offset / CENTER_X
    ) * 90

    angle = float(angle)

    # ==========================================
    # DIRECTION
    # ==========================================

    if angle > 50:

        direction = "sharp_left"

    elif angle > 10:

        direction = "slight_left"

    elif angle < -50:

        direction = "sharp_right"

    elif angle < -10:

        direction = "slight_right"

    else:

        direction = "straight"

    # ==========================================
    # SAVE
    # ==========================================

    gt[frame_id] = {

        "center_x": float(cx),

        "center_y": float(cy),

        "angle": angle,

        "direction": direction
    }

# ==========================================
# SAVE JSON
# ==========================================

with open(
    SAVE_PATH,
    "w"
) as f:

    json.dump(
        gt,
        f,
        indent=2
    )

print("Saved:", SAVE_PATH)

print()

print("Sample:")

for k in list(gt.keys())[:5]:

    print(k, gt[k])

Saved: /content/drive/MyDrive/KLTN/ground_truth_my_data_ver2.json

Sample:
000000 {'center_x': 306.5, 'center_y': 871.0999999999999, 'angle': 5.781249999999999, 'direction': 'straight'}
000020 {'center_x': 314.05, 'center_y': 870.75, 'angle': 8.140625000000004, 'direction': 'straight'}
000040 {'center_x': 320.6, 'center_y': 874.175, 'angle': 10.187500000000007, 'direction': 'slight_left'}
000060 {'center_x': 317.05, 'center_y': 857.625, 'angle': 9.078125000000004, 'direction': 'straight'}
000080 {'center_x': 291.565, 'center_y': 838.1875, 'angle': 1.1140624999999993, 'direction': 'straight'}


#### Tạo file zip -> Upload lên CVAT

In [ ]:
import os
import shutil
import zipfile

# ==========================================
# INPUT VIDEO FRAMES
# ==========================================

SRC_DIR = "/content/drive/MyDrive/KLTN/my_data_ver2/video_frames"

# ==========================================
# OUTPUT SAMPLE FOLDER
# ==========================================

SAMPLE_DIR = "/content/my_data_ver2"

# ==========================================
# OUTPUT ZIP
# ==========================================

ZIP_PATH = "/content/my_data_ver2.zip"

# ==========================================
# SAMPLE EVERY N FRAMES
# ==========================================

#STEP = 10 #SANPO
STEP = 20

# ==========================================
# CREATE FOLDER
# ==========================================

os.makedirs(SAMPLE_DIR, exist_ok=True)

# ==========================================
# GET ALL IMAGES
# ==========================================

files = sorted(os.listdir(SRC_DIR))

# ==========================================
# COPY SAMPLE FRAMES
# ==========================================

count = 0

for i, f in enumerate(files):

    # chỉ lấy file ảnh
    if not f.endswith((".png", ".jpg", ".jpeg")):
        continue

    # lấy mỗi STEP frame
    if i % STEP == 0:

        src_path = os.path.join(SRC_DIR, f)

        dst_path = os.path.join(SAMPLE_DIR, f)

        shutil.copy(src_path, dst_path)

        count += 1

print(f"Copied {count} sample frames")

# ==========================================
# CREATE ZIP
# ==========================================

with zipfile.ZipFile(
    ZIP_PATH,
    'w',
    zipfile.ZIP_DEFLATED
) as zipf:

    for root, dirs, files in os.walk(SAMPLE_DIR):

        for file in files:

            file_path = os.path.join(root, file)

            arcname = os.path.relpath(
                file_path,
                SAMPLE_DIR
            )

            zipf.write(file_path, arcname)

print("ZIP saved:", ZIP_PATH)

Copied 106 sample frames
ZIP saved: /content/my_data_ver2.zip
